# 24 - Bcc combined biocide phenotype-genome audit

This notebook tests whether multiple Bcc papers can form a second supervised benchmark closer to pharmaceutical-water and water-based-product risk.

Strict rule:

- row-level strain phenotype required;
- public genome match required;
- aggregate paper tables are context only;
- no inferred labels from species means, figures, or narrative text.

Primary row-level sources:

- Moore et al. 2009, DOI `10.1093/jac/dkn540`
- Kim/Wang et al. 2015, DOI `10.1007/s10295-015-1605-x`

Context source:

- Rushton et al. 2013, DOI `10.1128/AAC.00140-13`


In [ ]:
from __future__ import annotations
from pathlib import Path
import json
import subprocess
import sys
import pandas as pd

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

PHENOTYPES = PROJECT_ROOT / "data" / "manifests" / "bcc_combined_biocide_phenotype_rows.csv"
MATCHES = PROJECT_ROOT / "data" / "manifests" / "bcc_ncbi_assembly_match_candidates.csv"
MANIFEST = PROJECT_ROOT / "data" / "manifests" / "bcc_candidate_supervised_manifest.csv"
REPORT = PROJECT_ROOT / "reports" / "bcc_combined_phenotype_genome_audit.json"


## 1. Extract row-level phenotypes and query NCBI Assembly

In [ ]:
command = [sys.executable, str(PROJECT_ROOT / "scripts" / "audit_bcc_combined_phenotype_genome.py")]
print(" ".join(command))
subprocess.run(command, cwd=PROJECT_ROOT, check=True)

report = json.loads(REPORT.read_text())
report


## 2. Extracted phenotype rows

In [ ]:
phenotypes = pd.read_csv(PHENOTYPES)
print("Rows:", len(phenotypes))
print("Unique strains:", phenotypes["strain_id"].nunique())
display(pd.crosstab([phenotypes["study_key"], phenotypes["compound"]], phenotypes["endpoint"]))
display(phenotypes.head(30))


## 3. NCBI assembly match candidates

In [ ]:
matches = pd.read_csv(MATCHES)
print("Candidate rows:", len(matches))
if len(matches):
    print("Exact/contains matched query strains:", matches[matches["exact_norm_match"] | matches["contains_norm_match"]]["query_strain"].nunique())
    display(matches[[
        "query_strain", "assembly_accession", "organism", "biosample_accession",
        "assembly_strain", "exact_norm_match", "contains_norm_match", "ftp_path_refseq"
    ]].head(50))
else:
    print("No assembly matches returned.")


## 4. Candidate supervised manifest

In [ ]:
manifest = pd.read_csv(MANIFEST)
print("Manifest rows:", len(manifest))
if len(manifest):
    print("Unique matched strains:", manifest["strain_id"].nunique())
    display(pd.crosstab(manifest["compound"], manifest["endpoint"]))
    display(manifest.head(50))
else:
    print("No supervised manifest yet.")


## 5. Decision

If at least 20 strain-level phenotype records have credible public genome matches, this can become a small Bcc benchmark. Otherwise it remains biological support for the pharmaceutical-water prioritization paper.

Before modeling, manually audit every genome match. NCBI strain-name search can return related or ambiguous assemblies, especially for collection strain aliases.
